# ✈️ Flight Delay Detection and Prediction System (Notebook)

End-to-end workflow in one notebook: **synthetic 500-flight dataset → validation & preprocessing → compare 3 classifiers → regression for delay minutes → evaluation, charts → single & batch prediction**.

> **Important**
> * This is a *machine-learning prediction model*, **not** a live flight-tracking service.
> * The dataset is **synthetic** (simulated, fictional airlines). Scores here do **not** reflect real-world airline performance.
> * Target: `Delayed` = departure delay ≥ 15 min. Outcome columns (`Actual_Departure`, `Departure_Delay_Min`, `Delay_Status`) are never used as inputs.

Run the cells from top to bottom. Requirements: `pip install pandas numpy scikit-learn plotly joblib`.

## 1. Imports

In [1]:
import json, warnings
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
import sklearn
import plotly.express as px
import plotly.graph_objects as go
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
                             mean_squared_error, precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__)

scikit-learn 1.8.0 | pandas 3.0.2


## 2. Configuration
Column lists, thresholds, and the configurable risk bands (Low < 0.30 ≤ Medium < 0.60 ≤ High).

In [2]:
"""Central configuration: paths, column definitions and modelling constants."""



RANDOM_STATE = 42

# A flight is "Delayed" when its departure delay is at least this many minutes.
DELAY_THRESHOLD_MIN = 15
# Probability at/above which the classifier labels a flight "Delayed".
DECISION_THRESHOLD = 0.5
# Configurable risk bands (probability of delay). Low: p < low_max,
# High: p >= high_min, Medium: everything in between.
DEFAULT_RISK_THRESHOLDS = {"low_max": 0.30, "high_min": 0.60}

WEATHER_CONDITIONS = ["Clear", "Cloudy", "Rain", "Fog", "Snow", "Thunderstorm"]
TRAFFIC_LEVELS = ["Low", "Medium", "High"]

# Columns that must exist in any input used for prediction (single or batch).
REQUIRED_INPUT_COLUMNS = [
    "Airline", "Origin", "Destination", "Departure_Date",
    "Scheduled_Departure", "Scheduled_Arrival", "Distance_km",
    "Weather_Condition", "Temperature_C", "Wind_Speed_kmh",
    "Visibility_km", "Traffic_Level",
]
# Optional at prediction time. Previous_Delay_Min is only usable when the
# previous leg's delay is genuinely known before the flight departs.
OPTIONAL_INPUT_COLUMNS = ["Flight_ID", "Previous_Delay_Min"]
INPUT_FEATURE_COLUMNS = REQUIRED_INPUT_COLUMNS + ["Previous_Delay_Min"]

# Columns that describe the OUTCOME and must never be used as model inputs.
LEAKAGE_COLUMNS = ["Actual_Departure", "Departure_Delay_Min", "Delay_Status"]

# Dataset columns (training file).
DATASET_COLUMNS = [
    "Flight_ID", "Airline", "Origin", "Destination", "Departure_Date",
    "Scheduled_Departure", "Scheduled_Arrival", "Distance_km",
    "Weather_Condition", "Temperature_C", "Wind_Speed_kmh", "Visibility_km",
    "Traffic_Level", "Previous_Delay_Min", "Actual_Departure",
    "Departure_Delay_Min", "Delay_Status",
]

# Engineered feature groups fed to the ColumnTransformer.
CATEGORICAL_FEATURES = ["Airline", "Origin", "Destination", "Weather_Condition", "Traffic_Level"]
NUMERIC_FEATURES = [
    "Distance_km", "Temperature_C", "Wind_Speed_kmh", "Visibility_km",
    "Previous_Delay_Min", "Previous_Delay_Known", "Dep_Hour", "Day_Of_Week",
    "Month", "Is_Weekend", "Is_Peak_Hour", "Scheduled_Duration_Min",
]

MODEL_PATH = "flight_delay_model.joblib"
DATA_PATH = "data/flights_500.csv"

## 3. Generate the synthetic dataset (exactly 500 flights)
The label is derived from the delay minutes, never generated independently. A few blanks are injected on purpose to exercise missing-value handling.

In [3]:
import numpy as np
import pandas as pd


# Real airport codes with approximate coordinates (used only for distances).
AIRPORTS = {
    "JFK": (40.64, -73.78), "LAX": (33.94, -118.41), "ORD": (41.98, -87.90),
    "ATL": (33.64, -84.43), "DFW": (32.90, -97.04), "DEN": (39.86, -104.67),
    "SFO": (37.62, -122.38), "MIA": (25.79, -80.29),
}
# Fictional airlines with an invented "operational reliability" offset (minutes).
AIRLINES = {
    "SkyJet Airways": 2.0, "BlueHorizon Air": -2.5, "Meridian Airlines": 0.5,
    "Aurora Air": 4.0, "Pacific Wing": -1.0,
}
AIRLINE_WEIGHTS = [0.26, 0.22, 0.20, 0.16, 0.16]
# Extra delay minutes at congested origin airports.
AIRPORT_CONGESTION = {"JFK": 3.0, "ORD": 4.0, "ATL": 2.0, "SFO": 2.5}

WEATHER_EFFECT = {"Clear": 0.0, "Cloudy": 1.5, "Rain": 6.0, "Fog": 12.0, "Snow": 15.0, "Thunderstorm": 21.0}
TRAFFIC_EFFECT = {"Low": 0.0, "Medium": 4.0, "High": 11.0}


def _haversine_km(a: tuple[float, float], b: tuple[float, float]) -> float:
    lat1, lon1, lat2, lon2 = map(np.radians, (*a, *b))
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return float(2 * 6371.0 * np.arcsin(np.sqrt(h)))


def generate_dataset(n_rows: int = 500, seed: int = RANDOM_STATE) -> pd.DataFrame:
    """Create ``n_rows`` plausible synthetic flights with delay-driven labels."""
    rng = np.random.default_rng(seed)
    codes = list(AIRPORTS)
    airline_names = list(AIRLINES)
    rows = []

    start = pd.Timestamp("2025-01-01")
    for i in range(n_rows):
        origin = rng.choice(codes)
        destination = rng.choice([c for c in codes if c != origin])
        airline = rng.choice(airline_names, p=AIRLINE_WEIGHTS)
        date = start + pd.Timedelta(days=int(rng.integers(0, 365)))
        month = date.month

        hour_weights = np.array([4, 8, 9, 8, 6, 5, 5, 5, 6, 7, 8, 9, 8, 7, 6, 5, 3, 2, 2], dtype=float)
        hour = int(rng.choice(np.arange(5, 24), p=hour_weights / hour_weights.sum()))
        minute = int(rng.choice(np.arange(0, 60, 5)))
        dep_dt = date + pd.Timedelta(hours=hour, minutes=minute)

        distance = round(_haversine_km(AIRPORTS[origin], AIRPORTS[destination]) * 1.05)
        duration_min = int(round((distance / 800.0 * 60.0 + 30.0) / 5.0) * 5)
        arr_dt = dep_dt + pd.Timedelta(minutes=duration_min)

        temperature = 13 + 14 * np.sin(2 * np.pi * (month - 4) / 12) + rng.normal(0, 6)
        weather = rng.choice(
            list(WEATHER_EFFECT), p=[0.38, 0.27, 0.17, 0.07, 0.05, 0.06]
        )
        if weather == "Snow" and temperature > 3:
            weather = "Rain"
        if weather == "Thunderstorm" and temperature < 8:
            weather = "Rain"

        wind = rng.gamma(2.5, 7.0) + (14.0 if weather == "Thunderstorm" else 0.0)
        visibility = {
            "Fog": rng.uniform(0.5, 3.0), "Snow": rng.uniform(1.5, 6.0),
            "Rain": rng.uniform(4.0, 9.0), "Thunderstorm": rng.uniform(3.0, 8.0),
        }.get(weather, rng.uniform(8.0, 10.0))

        peak = (6 <= hour <= 9) or (16 <= hour <= 20)
        traffic = rng.choice(
            ["Low", "Medium", "High"], p=[0.15, 0.45, 0.40] if peak else [0.40, 0.42, 0.18]
        )

        # Delay carried over from the aircraft's previous leg (known before departure).
        if rng.random() < 0.55:
            prev = rng.normal(0, 4)
        else:
            prev = rng.exponential(22.0)
        prev = int(round(float(np.clip(prev + AIRLINES[airline] * 0.5, -10, 240))))

        mean_delay = (
            -9.0
            + WEATHER_EFFECT[weather]
            + TRAFFIC_EFFECT[traffic]
            + 0.45 * max(prev, 0)
            + 0.5 * max(wind - 25.0, 0.0)
            + 2.5 * max(5.0 - visibility, 0.0)
            + (5.0 if 17 <= hour <= 21 else 0.0)
            + (-3.0 if hour <= 7 else 0.0)
            + AIRLINES[airline]
            + AIRPORT_CONGESTION.get(origin, 0.0)
        )
        delay = mean_delay + rng.normal(0, 6.0) + rng.exponential(4.0) - 4.0
        delay = int(round(float(np.clip(delay, -15, 360))))
        actual_dt = dep_dt + pd.Timedelta(minutes=delay)

        rows.append({
            "Flight_ID": f"FL{i + 1:04d}",
            "Airline": airline,
            "Origin": origin,
            "Destination": destination,
            "Departure_Date": date.strftime("%Y-%m-%d"),
            "Scheduled_Departure": dep_dt.strftime("%H:%M"),
            "Scheduled_Arrival": arr_dt.strftime("%H:%M"),
            "Distance_km": int(distance),
            "Weather_Condition": weather,
            "Temperature_C": round(float(temperature), 1),
            "Wind_Speed_kmh": round(float(wind), 1),
            "Visibility_km": round(float(visibility), 1),
            "Traffic_Level": traffic,
            "Previous_Delay_Min": prev,
            "Actual_Departure": actual_dt.strftime("%Y-%m-%d %H:%M"),
            "Departure_Delay_Min": delay,
            "Delay_Status": "Delayed" if delay >= DELAY_THRESHOLD_MIN else "On-Time",
        })

    df = pd.DataFrame(rows, columns=DATASET_COLUMNS)

    # Realistic data gaps (blank sensor readings / unknown previous-leg delay).
    for column, count in (("Temperature_C", 5), ("Visibility_km", 4), ("Previous_Delay_Min", 8)):
        if n_rows > 50:
            idx = rng.choice(n_rows, size=count, replace=False)
            df[column] = df[column].astype(float)
            df.loc[idx, column] = np.nan
    df["Previous_Delay_Min"] = df["Previous_Delay_Min"].astype("Float64")
    return df

In [4]:
Path("data").mkdir(exist_ok=True)
raw = generate_dataset(500, seed=RANDOM_STATE)
raw.to_csv(DATA_PATH, index=False)
print(f"Saved {len(raw)} SYNTHETIC flights -> {DATA_PATH}")
raw.head()

Saved 500 SYNTHETIC flights -> data/flights_500.csv


,Flight_ID,Airline,Origin,Destination,Departure_Date,Scheduled_Departure,Scheduled_Arrival,Distance_km,Weather_Condition,Temperature_C,Wind_Speed_kmh,Visibility_km,Traffic_Level,Previous_Delay_Min,Actual_Departure,Departure_Delay_Min,Delay_Status
0,FL0001,BlueHorizon Air,JFK,SFO,2025-06-08,16:50,22:45,4360,Thunderstorm,13.4,30.5,7.6,High,-5.0,2025-06-08 17:16,26,Delayed
1,FL0002,Meridian Airlines,SFO,DEN,2025-03-02,11:45,14:15,1631,Rain,13.3,11.2,4.8,High,-3.0,2025-03-02 11:59,14,On-Time
2,FL0003,BlueHorizon Air,ATL,JFK,2025-05-01,16:10,18:15,1283,Fog,24.5,6.0,2.6,Low,-1.0,2025-05-01 16:22,12,On-Time
3,FL0004,Aurora Air,ATL,DEN,2025-07-22,15:25,18:25,2022,Clear,25.3,7.9,9.1,Low,0.0,2025-07-22 15:29,4,On-Time
4,FL0005,SkyJet Airways,MIA,DEN,2025-10-27,09:00,13:05,2886,Rain,13.9,11.2,8.1,Low,11.0,2025-10-27 09:01,1,On-Time


## 4. Validation, feature engineering and the shared pipeline
The same `build_pipeline` is used for training and prediction, so preprocessing can't drift between them.

In [5]:
"""Validation, cleaning, feature engineering and the shared sklearn pipeline.

The same ``build_pipeline`` object is used for training and inference, so
preprocessing can never drift between the two.
"""

from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


NUMERIC_RANGES = {
    "Distance_km": (1, 20000),
    "Temperature_C": (-60, 60),
    "Wind_Speed_kmh": (0, 300),
    "Visibility_km": (0, 100),
    "Previous_Delay_Min": (-60, 1440),
}
# Columns that may be blank in the TRAINING file (they are imputed later).
TRAINING_IMPUTABLE_COLUMNS = (
    "Temperature_C", "Wind_Speed_kmh", "Visibility_km",
    "Weather_Condition", "Traffic_Level",
)


class MissingColumnsError(ValueError):
    """Raised when required columns are absent from the input."""

    def __init__(self, columns: Iterable[str]):
        self.columns = list(columns)
        super().__init__("Missing required column(s): " + ", ".join(self.columns))


class InputValidationError(ValueError):
    """Raised when a single record fails validation. ``messages`` lists the problems."""

    def __init__(self, messages: list[str]):
        self.messages = list(messages)
        super().__init__("; ".join(self.messages))


# --------------------------------------------------------------------------- #
# Time helpers
# --------------------------------------------------------------------------- #
def time_to_minutes(series: pd.Series) -> pd.Series:
    """Convert 'HH:MM' / 'HH:MM:SS' strings (or time objects) to minutes after midnight."""
    s = series.astype(str).str.strip()
    s = s.str.replace(r"^(\d{1,2}:\d{2})$", r"\1:00", regex=True)
    td = pd.to_timedelta(s, errors="coerce")
    return td.dt.total_seconds() / 60.0


def _format_minutes(m: float):
    if pd.isna(m) or m < 0 or m >= 1440:
        return None
    return f"{int(m // 60):02d}:{int(m % 60):02d}"


# --------------------------------------------------------------------------- #
# Validation
# --------------------------------------------------------------------------- #
def validate_input_frame(
    df: pd.DataFrame, allow_missing: Iterable[str] = ()
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Validate prediction inputs row by row.

    Returns ``(valid_rows, errors)``. ``valid_rows`` keeps the original row
    positions as its index and contains normalised values. ``errors`` has one
    line per rejected row with human-readable issues.

    Raises ``MissingColumnsError`` if a required column does not exist.
    ``allow_missing`` lists columns whose blanks are tolerated (training only).
    """
    missing_cols = [c for c in REQUIRED_INPUT_COLUMNS if c not in df.columns]
    if missing_cols:
        raise MissingColumnsError(missing_cols)

    work = df.copy().reset_index(drop=True)
    n = len(work)
    reasons: list[list[str]] = [[] for _ in range(n)]
    allow = set(allow_missing)

    def flag(mask, message: str) -> None:
        arr = pd.Series(mask).fillna(False).to_numpy(dtype=bool)
        for i in np.flatnonzero(arr):
            reasons[i].append(message)

    # Free-text identifiers -------------------------------------------------
    for col in ("Airline", "Origin", "Destination"):
        s = work[col].astype("string").str.strip()
        blank = (s.isna() | (s == "")).fillna(True)
        flag(blank, f"{col} is missing")
        if col != "Airline":
            s = s.str.upper()
        work[col] = s.astype(object).where(~blank, np.nan)

    # Controlled vocabularies -----------------------------------------------
    for col, allowed in (("Weather_Condition", WEATHER_CONDITIONS), ("Traffic_Level", TRAFFIC_LEVELS)):
        s = work[col].astype("string").str.strip().str.title()
        miss = (s.isna() | (s == "")).fillna(True)
        invalid = (~miss) & (~s.isin(allowed)).fillna(True)
        if col not in allow:
            flag(miss, f"{col} is missing")
        flag(invalid, f"{col} must be one of {allowed}")
        work[col] = s.astype(object).where(~(miss | invalid), np.nan)

    # Numeric fields ----------------------------------------------------------
    for col, (lo, hi) in NUMERIC_RANGES.items():
        if col not in work.columns:
            work[col] = np.nan
        raw = work[col].replace(r"^\s*$", np.nan, regex=True)
        num = pd.to_numeric(raw, errors="coerce")
        missing = raw.isna()
        unparsable = (~missing) & num.isna()
        if col != "Previous_Delay_Min" and col not in allow:
            flag(missing, f"{col} is missing")
        flag(unparsable, f"{col} is not a number")
        flag(num.notna() & ((num < lo) | (num > hi)), f"{col} must be between {lo} and {hi}")
        work[col] = num.astype(float)

    # Date and times ----------------------------------------------------------
    date = pd.to_datetime(
        work["Departure_Date"].astype(str).str.strip(), format="ISO8601", errors="coerce"
    )
    flag(date.isna(), "Departure_Date is missing or not a valid date (use YYYY-MM-DD)")
    work["Departure_Date"] = date.dt.strftime("%Y-%m-%d")

    for col in ("Scheduled_Departure", "Scheduled_Arrival"):
        mins = time_to_minutes(work[col])
        formatted = [_format_minutes(m) for m in mins]
        flag([f is None for f in formatted],
             f"{col} is missing or not a valid time (use HH:MM, 00:00-23:59)")
        work[col] = pd.Series(formatted, index=work.index, dtype=object)

    # Cross-field and identity checks -----------------------------------------
    flag(work["Origin"].notna() & (work["Origin"] == work["Destination"]),
         "Origin and Destination must be different airports")
    if "Flight_ID" in work.columns:
        flag(work["Flight_ID"].notna() & work["Flight_ID"].duplicated(keep="first"),
             "Duplicate Flight_ID")

    bad = [i for i, r in enumerate(reasons) if r]
    flight_ids = work["Flight_ID"] if "Flight_ID" in work.columns else pd.Series([None] * n)
    errors = pd.DataFrame({
        "Row": [i + 1 for i in bad],
        "Flight_ID": [flight_ids.iloc[i] for i in bad],
        "Issues": ["; ".join(reasons[i]) for i in bad],
    })
    valid = work.drop(index=bad)
    return valid, errors


# --------------------------------------------------------------------------- #
# Feature engineering and pipeline
# --------------------------------------------------------------------------- #
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Turn validated flight inputs into model features (no outcome columns used)."""
    out = pd.DataFrame(index=df.index)
    dep = time_to_minutes(df["Scheduled_Departure"])
    arr = time_to_minutes(df["Scheduled_Arrival"])
    date = pd.to_datetime(df["Departure_Date"].astype(str), format="ISO8601", errors="coerce")

    for col in ("Distance_km", "Temperature_C", "Wind_Speed_kmh", "Visibility_km"):
        out[col] = pd.to_numeric(df[col], errors="coerce").astype(float)
    prev = (pd.to_numeric(df["Previous_Delay_Min"], errors="coerce").astype(float)
            if "Previous_Delay_Min" in df.columns else pd.Series(np.nan, index=df.index))
    out["Previous_Delay_Min"] = prev
    out["Previous_Delay_Known"] = prev.notna().astype(float)
    out["Dep_Hour"] = np.floor(dep / 60.0)
    out["Day_Of_Week"] = date.dt.dayofweek.astype(float)
    out["Month"] = date.dt.month.astype(float)
    out["Is_Weekend"] = (date.dt.dayofweek >= 5).astype(float)
    hour = out["Dep_Hour"]
    out["Is_Peak_Hour"] = (hour.between(6, 9) | hour.between(16, 20)).astype(float)
    out["Scheduled_Duration_Min"] = (arr - dep) % 1440  # handles overnight arrivals

    for col in CATEGORICAL_FEATURES:
        s = df[col]
        out[col] = s.astype(object).where(s.notna(), np.nan)
    return out[NUMERIC_FEATURES + CATEGORICAL_FEATURES]


def build_preprocessor() -> ColumnTransformer:
    numeric = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric, NUMERIC_FEATURES),
        ("cat", categorical, CATEGORICAL_FEATURES),
    ])


def build_pipeline(estimator) -> Pipeline:
    """Feature engineering + preprocessing + estimator as ONE fitted object."""
    return Pipeline([
        ("features", FunctionTransformer(engineer_features, validate=False)),
        ("preprocess", build_preprocessor()),
        ("model", estimator),
    ])


def assert_no_leakage(columns: Iterable[str]) -> None:
    leaked = sorted(set(columns) & set(LEAKAGE_COLUMNS))
    if leaked:
        raise ValueError(f"Target leakage: outcome columns used as features: {leaked}")


# --------------------------------------------------------------------------- #
# Training-data loading and cleaning
# --------------------------------------------------------------------------- #
def load_dataset(path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found at {path}. Generate it with: python generate_dataset.py"
        )
    return pd.read_csv(path)


def clean_training_data(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Deduplicate, validate and label the training data. Returns (clean_df, report)."""
    needed = ["Flight_ID", "Departure_Delay_Min"]
    absent = [c for c in needed if c not in df.columns]
    if absent:
        raise MissingColumnsError(absent)

    report = {"rows_in": int(len(df))}
    df = df.copy()
    dup_mask = df["Flight_ID"].duplicated(keep="first")
    report["duplicate_ids_removed"] = int(dup_mask.sum())
    df = df[~dup_mask].reset_index(drop=True)

    df["Departure_Delay_Min"] = pd.to_numeric(df["Departure_Delay_Min"], errors="coerce")
    no_target = df["Departure_Delay_Min"].isna()
    report["rows_missing_target"] = int(no_target.sum())
    df = df[~no_target].reset_index(drop=True)

    valid, errors = validate_input_frame(df, allow_missing=TRAINING_IMPUTABLE_COLUMNS)
    report["rows_invalid_dropped"] = int(len(errors))
    clean = df.loc[valid.index, ["Flight_ID", "Departure_Delay_Min"]].copy()
    clean = pd.concat([clean.reset_index(drop=True), valid.drop(columns=["Flight_ID", "Departure_Delay_Min"], errors="ignore").reset_index(drop=True)], axis=1)

    # The label is always derived from the delay minutes, never trusted blindly.
    derived = np.where(clean["Departure_Delay_Min"] >= DELAY_THRESHOLD_MIN, "Delayed", "On-Time")
    if "Delay_Status" in df.columns:
        original = df.loc[valid.index, "Delay_Status"].reset_index(drop=True)
        report["labels_inconsistent_with_minutes"] = int((original.astype(str) != derived).sum())
    clean["Delay_Status"] = derived
    report["rows_out"] = int(len(clean))
    report["missing_values_imputed"] = {
        c: int(clean[c].isna().sum()) for c in INPUT_FEATURE_COLUMNS if clean[c].isna().any()
    }
    return clean, report


def make_xy(clean: pd.DataFrame):
    """Split into features X, binary target y (1 = Delayed) and delay minutes."""
    X = clean[INPUT_FEATURE_COLUMNS].copy()
    assert_no_leakage(X.columns)
    y = (clean["Departure_Delay_Min"] >= DELAY_THRESHOLD_MIN).astype(int)
    return X, y, clean["Departure_Delay_Min"].astype(float)


def add_scheduled_hour(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Dep_Hour"] = np.floor(time_to_minutes(out["Scheduled_Departure"]) / 60.0)
    return out


def dataset_quality_report(df: pd.DataFrame) -> dict:
    return {
        "rows": int(len(df)),
        "columns": int(df.shape[1]),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_flight_ids": int(df["Flight_ID"].duplicated().sum()) if "Flight_ID" in df else 0,
        "missing_by_column": df.isna().sum().astype(int).to_dict(),
    }

## 5. Evaluation helpers

In [6]:
"""Metric calculation, model selection and feature-importance extraction."""

import numpy as np
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score, mean_absolute_error,
    mean_squared_error, precision_score, recall_score, roc_auc_score, roc_curve,
)



def classification_metrics(y_true, y_pred, y_proba=None) -> dict:
    """Compute every classification metric from real predictions.

    ``roc_auc`` and the ROC curve are ``None`` unless both classes are present
    in ``y_true`` and probability scores were supplied.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": None,
        "confusion_matrix": cm.tolist(),  # [[TN, FP], [FN, TP]]
        "support": {"on_time": int((y_true == 0).sum()), "delayed": int((y_true == 1).sum())},
        "roc_curve": None,
    }
    if y_proba is not None and len(np.unique(y_true)) == 2:
        proba = np.asarray(y_proba, dtype=float)
        result["roc_auc"] = float(roc_auc_score(y_true, proba))
        fpr, tpr, _ = roc_curve(y_true, proba)
        result["roc_curve"] = {
            "fpr": np.round(fpr, 4).tolist(),
            "tpr": np.round(tpr, 4).tolist(),
        }
    return result


def regression_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def select_best_model(results: dict) -> str:
    """Pick the model with the highest mean cross-validated F1 on TRAINING data.

    Ties are broken by cross-validated ROC-AUC. The held-out test set is never
    used for selection, only for the final report.
    """
    if not results:
        raise ValueError("No model results to select from.")
    return max(
        results,
        key=lambda name: (results[name]["cv_f1_mean"], results[name].get("cv_roc_auc_mean") or 0.0),
    )


def _clean_name(raw: str) -> str:
    return raw.split("__", 1)[1] if "__" in raw else raw


def _group_name(feature: str) -> str:
    for col in CATEGORICAL_FEATURES:
        if feature.startswith(col + "_"):
            return col
    return feature


def extract_feature_importance(pipeline, top_n: int = 15) -> dict | None:
    """Return importances for tree models (``feature_importances_``) or linear
    models (absolute coefficients on standardised inputs). ``None`` if unsupported."""
    model = pipeline.named_steps["model"]
    names = [_clean_name(n) for n in pipeline.named_steps["preprocess"].get_feature_names_out()]
    if hasattr(model, "feature_importances_"):
        values = np.asarray(model.feature_importances_, dtype=float)
        kind = "Impurity-based importance"
    elif hasattr(model, "coef_"):
        values = np.abs(np.asarray(model.coef_, dtype=float)).ravel()
        kind = "Absolute coefficient (standardised inputs)"
    else:
        return None

    detail = sorted(zip(names, values), key=lambda t: t[1], reverse=True)
    grouped: dict[str, float] = {}
    for name, value in detail:
        grouped[_group_name(name)] = grouped.get(_group_name(name), 0.0) + float(value)
    total = sum(grouped.values()) or 1.0
    grouped_list = sorted(
        ({"feature": k, "importance": v / total} for k, v in grouped.items()),
        key=lambda d: d["importance"], reverse=True,
    )
    return {
        "type": kind,
        "detail": [{"feature": n, "importance": float(v)} for n, v in detail[:top_n]],
        "grouped": grouped_list,
    }

## 6. Prediction helpers (single/batch, risk bands, explanations)

In [7]:
"""Model loading, single/batch prediction, risk bands and explanations."""

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn


REQUIRED_BUNDLE_KEYS = ("classifier", "model_name", "decision_threshold", "baselines", "categories")


class ModelNotFoundError(FileNotFoundError):
    """Raised when the trained model artifact is missing."""


class ModelLoadError(RuntimeError):
    """Raised when the model artifact exists but cannot be used."""


def load_model_bundle(path=MODEL_PATH) -> dict:
    path = Path(path)
    if not path.exists():
        raise ModelNotFoundError(
            f"Trained model not found at '{path}'. Train it first with: python train_model.py"
        )
    try:
        bundle = joblib.load(path)
    except Exception as exc:  # corrupt file, version mismatch, missing module...
        raise ModelLoadError(
            f"Could not load '{path}' ({exc}). Re-train the model with: python train_model.py"
        ) from exc
    if not isinstance(bundle, dict) or any(k not in bundle for k in REQUIRED_BUNDLE_KEYS):
        raise ModelLoadError(
            f"'{path}' is not a valid model bundle. Re-train the model with: python train_model.py"
        )
    saved = str(bundle.get("sklearn_version", "unknown"))
    if saved.split(".")[:2] != sklearn.__version__.split(".")[:2]:
        raise ModelLoadError(
            f"The saved model was trained with scikit-learn {saved}, but scikit-learn "
            f"{sklearn.__version__} is installed, so it may not work correctly. "
            "Re-train it on this machine with: python train_model.py"
        )
    return bundle


def validate_risk_thresholds(thresholds: dict | None) -> dict:
    t = dict(DEFAULT_RISK_THRESHOLDS if thresholds is None else thresholds)
    low, high = t.get("low_max"), t.get("high_min")
    if low is None or high is None or not (0.0 <= low < high <= 1.0):
        raise ValueError("Risk thresholds must satisfy 0 <= low_max < high_min <= 1.")
    return {"low_max": float(low), "high_min": float(high)}


def risk_category(probability: float, thresholds: dict | None = None) -> str:
    """Low: p < low_max; High: p >= high_min; otherwise Medium."""
    t = validate_risk_thresholds(thresholds)
    if probability < t["low_max"]:
        return "Low"
    if probability >= t["high_min"]:
        return "High"
    return "Medium"


def _delay_probability(pipeline, X: pd.DataFrame) -> np.ndarray:
    proba = pipeline.predict_proba(X)
    classes = list(pipeline.classes_)
    if 1 not in classes:
        return np.zeros(len(X))
    return proba[:, classes.index(1)]


def _estimated_minutes(bundle: dict, X: pd.DataFrame):
    regressor = bundle.get("regressor")
    if regressor is None:
        return [None] * len(X)
    return np.round(np.clip(regressor.predict(X), 0, None), 1)


def predict_dataframe(bundle: dict, df: pd.DataFrame, risk_thresholds: dict | None = None):
    """Validate and predict many flights.

    Returns ``(results, errors)``. Rows that fail validation are excluded from
    ``results`` and described in ``errors``. Raises ``MissingColumnsError`` if
    required columns are absent. Actual-delay/actual-departure columns are
    never read.
    """
    thresholds = validate_risk_thresholds(risk_thresholds)
    valid, errors = validate_input_frame(df)
    base_cols = (["Flight_ID"] if "Flight_ID" in valid.columns else []) + INPUT_FEATURE_COLUMNS
    if valid.empty:
        empty = pd.DataFrame(columns=["Row"] + base_cols + [
            "Predicted_Status", "Delay_Probability", "Risk_Category", "Estimated_Delay_Min"])
        return empty, errors

    X = valid[INPUT_FEATURE_COLUMNS]
    probability = _delay_probability(bundle["classifier"], X)
    status = np.where(probability >= bundle["decision_threshold"], "Delayed", "On-Time")
    results = valid[base_cols].copy()
    results.insert(0, "Row", valid.index + 1)
    results["Predicted_Status"] = status
    results["Delay_Probability"] = np.round(probability, 4)
    results["Risk_Category"] = [risk_category(p, thresholds) for p in probability]
    results["Estimated_Delay_Min"] = _estimated_minutes(bundle, X)
    return results.reset_index(drop=True), errors


# --------------------------------------------------------------------------- #
# Explanation (occlusion against a "typical flight")
# --------------------------------------------------------------------------- #
def explain_prediction(bundle: dict, row: pd.DataFrame, top_n: int = 5) -> list[dict]:
    """Estimate how much each key input moves the delay probability.

    For each factor, the input is replaced by its typical (median/mode) value
    from the training data and the change is recorded both in probability
    points and as an odds multiplier (log-odds are used for ranking because raw
    probabilities saturate near 0% / 100%). This is a local sensitivity
    estimate, not proof of causation.
    """
    base = bundle["baselines"]
    row = row[INPUT_FEATURE_COLUMNS].iloc[[0]].copy()
    p_actual = float(_delay_probability(bundle["classifier"], row)[0])

    variants, meta = [], []

    def add(name: str, value_label, baseline_label, mutate):
        counterfactual = row.copy()
        mutate(counterfactual)
        variants.append(counterfactual)
        meta.append((name, value_label, baseline_label))

    r = row.iloc[0]
    for col, label in (("Weather_Condition", "Weather"), ("Traffic_Level", "Traffic level"), ("Airline", "Airline")):
        if r[col] != base[col]:
            add(label, r[col], base[col], lambda d, c=col: d.__setitem__(c, base[c]))
    for col, label, unit in (
        ("Wind_Speed_kmh", "Wind speed", "km/h"), ("Visibility_km", "Visibility", "km"),
        ("Temperature_C", "Temperature", "°C"), ("Distance_km", "Distance", "km"),
    ):
        if pd.notna(r[col]) and abs(float(r[col]) - base[col]) > 1e-9:
            add(label, f"{float(r[col]):g} {unit}", f"{base[col]:.4g} {unit}",
                lambda d, c=col: d.__setitem__(c, base[c]))
    if pd.notna(r["Previous_Delay_Min"]) and abs(float(r["Previous_Delay_Min"]) - base["Previous_Delay_Min"]) > 1e-9:
        add("Previous delay", f"{float(r['Previous_Delay_Min']):g} min", f"{base['Previous_Delay_Min']:.4g} min",
            lambda d: d.__setitem__("Previous_Delay_Min", base["Previous_Delay_Min"]))

    dep = float(time_to_minutes(pd.Series([r["Scheduled_Departure"]])).iloc[0])
    base_dep = float(time_to_minutes(pd.Series([base["Scheduled_Departure"]])).iloc[0])
    if abs(dep - base_dep) > 1e-9:
        arr = float(time_to_minutes(pd.Series([r["Scheduled_Arrival"]])).iloc[0])

        def shift_departure(d, dep=dep, arr=arr, base_dep=base_dep):
            new_arr = (arr + (base_dep - dep)) % 1440
            d["Scheduled_Departure"] = f"{int(base_dep // 60):02d}:{int(base_dep % 60):02d}"
            d["Scheduled_Arrival"] = f"{int(new_arr // 60):02d}:{int(new_arr % 60):02d}"

        add("Departure time", r["Scheduled_Departure"], base["Scheduled_Departure"], shift_departure)

    if not variants:
        return []
    p_counter = _delay_probability(bundle["classifier"], pd.concat(variants, ignore_index=True))

    def logit(p: float) -> float:
        p = min(max(float(p), 1e-6), 1 - 1e-6)
        return float(np.log(p / (1 - p)))

    factors = []
    for (name, value, typical), p_cf in zip(meta, p_counter):
        shift = logit(p_actual) - logit(p_cf)  # change in log-odds vs. a typical flight
        factors.append({
            "factor": name,
            "value": str(value),
            "typical_value": str(typical),
            "impact_points": round((p_actual - float(p_cf)) * 100.0, 1),
            "odds_multiplier": round(float(np.exp(shift)), 2),
            "log_odds_shift": round(shift, 3),
            "direction": "raises" if shift > 0 else "lowers",
        })
    factors.sort(key=lambda f: abs(f["log_odds_shift"]), reverse=True)
    return factors[:top_n]


def predict_single(bundle: dict, record: dict, risk_thresholds: dict | None = None) -> dict:
    """Predict one flight. Raises ``InputValidationError`` with readable messages."""
    absent = [c for c in INPUT_FEATURE_COLUMNS if c != "Previous_Delay_Min" and c not in record]
    if absent:
        raise InputValidationError([f"Missing field: {c}" for c in absent])
    frame = pd.DataFrame([record])
    try:
        results, errors = predict_dataframe(bundle, frame, risk_thresholds)
    except MissingColumnsError as exc:
        raise InputValidationError([str(exc)]) from exc
    if not errors.empty:
        raise InputValidationError(errors["Issues"].iloc[0].split("; "))

    out = results.iloc[0].to_dict()
    valid, _ = validate_input_frame(frame)
    out["Explanation"] = explain_prediction(bundle, valid)
    return out

## 7. Chart helpers (Plotly)

In [8]:
"""Plotly figure builders used by the dashboard."""

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


NAVY = "#0B2545"
SKY = "#1F78C8"
TEAL = "#2A9D8F"
ALERT = "#E4572E"
AMBER = "#F2A541"
GRID = "#E3EAF3"
STATUS_COLORS = {"On-Time": TEAL, "Delayed": ALERT}


def _style(fig: go.Figure, title: str | None = None, height: int = 380) -> go.Figure:
    fig.update_layout(
        title=dict(text=title, font=dict(size=16, color=NAVY)) if title else None,
        height=height,
        margin=dict(l=30, r=20, t=60 if title else 20, b=40),
        plot_bgcolor="white",
        paper_bgcolor="white",
        font=dict(family="Inter, Segoe UI, Arial, sans-serif", color=NAVY),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.update_xaxes(gridcolor=GRID, zeroline=False)
    fig.update_yaxes(gridcolor=GRID, zeroline=False)
    return fig


def _delay_rate_table(df: pd.DataFrame, column: str) -> pd.DataFrame:
    g = df.groupby(column, observed=True)["Delay_Status"]
    table = pd.DataFrame({
        "Flights": g.size(),
        "Delayed": g.apply(lambda s: int((s == "Delayed").sum())),
    }).reset_index()
    table["Delay_Rate_%"] = (table["Delayed"] / table["Flights"] * 100).round(1)
    return table


# --------------------------------------------------------------------------- #
# Analytics
# --------------------------------------------------------------------------- #
def status_chart(df: pd.DataFrame) -> go.Figure:
    counts = df["Delay_Status"].value_counts().reindex(["On-Time", "Delayed"]).fillna(0)
    fig = go.Figure(go.Pie(
        labels=counts.index, values=counts.values, hole=0.55,
        marker=dict(colors=[STATUS_COLORS[k] for k in counts.index]),
        textinfo="label+percent",
    ))
    return _style(fig, "Delayed vs on-time flights", 340)


def delay_rate_chart(df: pd.DataFrame, column: str, title: str, order: list | None = None) -> go.Figure:
    table = _delay_rate_table(df, column)
    if order:
        table[column] = pd.Categorical(table[column], categories=order, ordered=True)
        table = table.sort_values(column)
    else:
        table = table.sort_values("Delay_Rate_%", ascending=False)
    fig = px.bar(
        table, x=column, y="Delay_Rate_%", text="Delay_Rate_%",
        hover_data={"Flights": True, "Delayed": True},
        color="Delay_Rate_%", color_continuous_scale=[TEAL, AMBER, ALERT],
    )
    fig.update_traces(texttemplate="%{text:.0f}%", textposition="outside")
    fig.update_layout(coloraxis_showscale=False, yaxis_title="Delayed flights (%)")
    fig.update_yaxes(range=[0, max(100, table["Delay_Rate_%"].max() * 1.15)])
    return _style(fig, title)


def delay_by_hour_chart(df: pd.DataFrame) -> go.Figure:
    work = add_scheduled_hour(df).dropna(subset=["Dep_Hour"])
    work["Dep_Hour"] = work["Dep_Hour"].astype(int)
    table = _delay_rate_table(work, "Dep_Hour")
    fig = go.Figure()
    fig.add_bar(x=table["Dep_Hour"], y=table["Flights"], name="Flights", marker_color=GRID, yaxis="y2")
    fig.add_scatter(
        x=table["Dep_Hour"], y=table["Delay_Rate_%"], name="Delay rate (%)",
        mode="lines+markers", line=dict(color=ALERT, width=3),
    )
    fig.update_layout(
        xaxis=dict(title="Scheduled departure hour", dtick=1),
        yaxis=dict(title="Delayed flights (%)", range=[0, 105]),
        yaxis2=dict(title="Flights", overlaying="y", side="right", showgrid=False),
    )
    return _style(fig, "Delay rate by scheduled departure hour")


def delay_distribution_chart(df: pd.DataFrame) -> go.Figure:
    fig = px.histogram(
        df, x="Departure_Delay_Min", color="Delay_Status", nbins=40,
        color_discrete_map=STATUS_COLORS, barmode="overlay", opacity=0.8,
    )
    fig.add_vline(x=DELAY_THRESHOLD_MIN, line_dash="dash", line_color=NAVY,
                  annotation_text=f"{DELAY_THRESHOLD_MIN} min threshold")
    fig.update_layout(xaxis_title="Departure delay (minutes; negative = early)", yaxis_title="Flights")
    return _style(fig, "Delay duration distribution")


def origin_destination_heatmap(df: pd.DataFrame) -> go.Figure:
    work = df.assign(Delayed=(df["Delay_Status"] == "Delayed").astype(int))
    rate = work.pivot_table(index="Origin", columns="Destination", values="Delayed", aggfunc="mean") * 100
    count = work.pivot_table(index="Origin", columns="Destination", values="Delayed", aggfunc="size")
    fig = go.Figure(go.Heatmap(
        z=rate.values, x=list(rate.columns), y=list(rate.index),
        customdata=count.reindex(index=rate.index, columns=rate.columns).values,
        colorscale=[[0, TEAL], [0.5, AMBER], [1, ALERT]], zmin=0, zmax=100,
        hovertemplate="%{y} → %{x}<br>Delay rate: %{z:.0f}%<br>Flights: %{customdata}<extra></extra>",
        colorbar=dict(title="Delay %"),
    ))
    fig.update_layout(xaxis_title="Destination", yaxis_title="Origin")
    return _style(fig, "Delay rate by route (origin → destination)", 440)


# --------------------------------------------------------------------------- #
# Model performance
# --------------------------------------------------------------------------- #
def model_comparison_chart(metrics: dict) -> go.Figure:
    rows = []
    for name, res in metrics["models"].items():
        for key in ("accuracy", "precision", "recall", "f1"):
            rows.append({"Model": name, "Metric": key.upper() if key == "f1" else key.title(),
                         "Score": res["test"][key]})
    fig = px.bar(pd.DataFrame(rows), x="Metric", y="Score", color="Model", barmode="group",
                 color_discrete_sequence=[SKY, TEAL, AMBER])
    fig.update_yaxes(range=[0, 1])
    return _style(fig, "Test-set metrics by model")


def confusion_matrix_chart(cm: list[list[int]], title: str = "Confusion matrix") -> go.Figure:
    labels = ["On-Time", "Delayed"]
    fig = px.imshow(
        np.array(cm), x=[f"Pred {l}" for l in labels], y=[f"Actual {l}" for l in labels],
        text_auto=True, color_continuous_scale=["#EAF2FB", SKY], aspect="auto",
    )
    fig.update_layout(coloraxis_showscale=False)
    return _style(fig, title, 340)


def roc_chart(metrics: dict, highlight: str | None = None) -> go.Figure | None:
    fig = go.Figure()
    drawn = False
    palette = [SKY, TEAL, AMBER]
    for i, (name, res) in enumerate(metrics["models"].items()):
        curve = res["test"].get("roc_curve")
        if not curve:
            continue
        drawn = True
        auc = res["test"]["roc_auc"]
        fig.add_scatter(
            x=curve["fpr"], y=curve["tpr"], mode="lines",
            name=f"{name} (AUC {auc:.2f})",
            line=dict(color=palette[i % 3], width=4 if name == highlight else 2),
        )
    if not drawn:
        return None
    fig.add_scatter(x=[0, 1], y=[0, 1], mode="lines", name="Chance",
                    line=dict(color="#9AA7B8", dash="dash"))
    fig.update_layout(xaxis_title="False positive rate", yaxis_title="True positive rate")
    return _style(fig, "ROC curves (test set)", 380)


def feature_importance_chart(items: list[dict], title: str) -> go.Figure:
    df = pd.DataFrame(items).sort_values("importance")
    fig = px.bar(df, x="importance", y="feature", orientation="h", color_discrete_sequence=[SKY])
    fig.update_layout(xaxis_title="Importance", yaxis_title=None)
    return _style(fig, title, max(320, 28 * len(df) + 100))


# --------------------------------------------------------------------------- #
# Prediction page
# --------------------------------------------------------------------------- #
def risk_gauge(probability: float, thresholds: dict | None = None) -> go.Figure:
    t = thresholds or DEFAULT_RISK_THRESHOLDS
    low, high = t["low_max"] * 100, t["high_min"] * 100
    fig = go.Figure(go.Indicator(
        mode="gauge+number", value=probability * 100,
        number=dict(suffix="%", font=dict(color=NAVY)),
        title=dict(text="Estimated probability of delay"),
        gauge=dict(
            axis=dict(range=[0, 100]), bar=dict(color=NAVY),
            steps=[
                dict(range=[0, low], color="#BFE6DF"),
                dict(range=[low, high], color="#FBE0B5"),
                dict(range=[high, 100], color="#F6B9A8"),
            ],
        ),
    ))
    return _style(fig, None, 260)


def factor_impact_chart(factors: list[dict]) -> go.Figure:
    df = pd.DataFrame(factors).sort_values("log_odds_shift")
    df["label"] = df["factor"] + ": " + df["value"]
    fig = go.Figure(go.Bar(
        x=df["log_odds_shift"], y=df["label"], orientation="h",
        marker_color=[ALERT if v > 0 else TEAL for v in df["log_odds_shift"]],
        customdata=df[["odds_multiplier", "typical_value"]].values,
        hovertemplate="%{y}<br>Odds ×%{customdata[0]} vs typical (%{customdata[1]})<extra></extra>",
    ))
    fig.update_layout(xaxis_title="Shift in log-odds of delay vs a typical flight (right = more delay risk)")
    return _style(fig, "Key input factors", max(260, 55 * len(df) + 110))

## 8. Load, clean and split
Duplicates removed, labels recomputed from delay minutes, stratified 80/20 split.

In [9]:
data = pd.read_csv(DATA_PATH)
clean, report = clean_training_data(data)
X, y, delay_minutes = make_xy(clean)
print(json.dumps(report, indent=2))
print("Delayed share:", round(y.mean(), 3))

X_train, X_test, y_train, y_test, d_train, d_test = train_test_split(
    X, y, delay_minutes, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print("Train:", X_train.shape, "Test:", X_test.shape)

{
  "rows_in": 500,
  "duplicate_ids_removed": 0,
  "rows_missing_target": 0,
  "rows_invalid_dropped": 0,
  "labels_inconsistent_with_minutes": 0,
  "rows_out": 500,
  "missing_values_imputed": {
    "Temperature_C": 5,
    "Visibility_km": 4,
    "Previous_Delay_Min": 8
  }
}
Delayed share: 0.336
Train: (400, 13) Test: (100, 13)


## 9. Train and compare three classifiers
Selection criterion: **highest mean 5-fold cross-validated F1 on the training set** (F1 balances precision and recall, so it isn't fooled by class imbalance like accuracy). The test set is used only for the final report.

In [10]:
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=3,
                                                    subsample=0.8, random_state=RANDOM_STATE),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results, fitted = {}, {}
for name, estimator in candidates.items():
    pipe = build_pipeline(estimator)
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["f1", "roc_auc"])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= DECISION_THRESHOLD).astype(int)
    results[name] = {"cv_f1_mean": float(scores["test_f1"].mean()),
                     "cv_roc_auc_mean": float(scores["test_roc_auc"].mean()),
                     "test": classification_metrics(y_test, pred, proba)}
    fitted[name] = pipe

best_name = select_best_model(results)
metrics = {"models": results}
summary = pd.DataFrame({
    n: {"CV F1 (train)": r["cv_f1_mean"], **{k.title(): r["test"][k] for k in ("accuracy", "precision", "recall", "f1", "roc_auc")}}
    for n, r in results.items()}).T.round(3)
print("Selected model:", best_name)
summary

Selected model: Logistic Regression


,CV F1 (train),Accuracy,Precision,Recall,F1,Roc_Auc
Logistic Regression,0.788,0.76,0.625,0.735,0.676,0.865
Random Forest,0.709,0.77,0.649,0.706,0.676,0.844
Gradient Boosting,0.740,0.81,0.727,0.706,0.716,0.868


### Confusion matrix and ROC curves (test set)

In [11]:
confusion_matrix_chart(results[best_name]["test"]["confusion_matrix"], f"Confusion matrix - {best_name}").show()
roc_chart(metrics, highlight=best_name).show()
model_comparison_chart(metrics).show()

### Feature importance

In [12]:
best_pipeline = fitted[best_name]
importance = extract_feature_importance(best_pipeline)
print(importance["type"])
feature_importance_chart(importance["grouped"], "Importance by input (normalised)").show()
feature_importance_chart(importance["detail"], "Top encoded features").show()

Absolute coefficient (standardised inputs)


## 10. Regression: estimate delay minutes
Target is delay minutes **clipped at 0** (early departures count as 0) and predictions are clipped to be non-negative. Evaluated with MAE / RMSE against an "always predict the mean" baseline.

In [13]:
regressor = build_pipeline(GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3,
                                                     subsample=0.8, random_state=RANDOM_STATE))
regressor.fit(X_train, d_train.clip(lower=0))
reg_pred = np.clip(regressor.predict(X_test), 0, None)
reg_true = d_test.clip(lower=0)
reg_metrics = regression_metrics(reg_true, reg_pred)
reg_metrics["baseline_mae_predict_mean"] = regression_metrics(reg_true, np.full(len(reg_true), d_train.clip(lower=0).mean()))["mae"]
print({k: round(v, 2) for k, v in reg_metrics.items()})

{'mae': 5.24, 'rmse': 6.9, 'baseline_mae_predict_mean': 9.5}


## 11. Analytics
These show **associations** in the dataset; they do not establish causation.

In [14]:
df = data.copy()
status_chart(df).show()
delay_rate_chart(df, "Airline", "Delay percentage by airline").show()
delay_by_hour_chart(df).show()
delay_rate_chart(df, "Weather_Condition", "Delay patterns by weather", WEATHER_CONDITIONS).show()
delay_rate_chart(df, "Traffic_Level", "Delay patterns by traffic level", TRAFFIC_LEVELS).show()
delay_distribution_chart(df).show()
origin_destination_heatmap(df).show()

## 12. Save the model bundle (Joblib)
Trained and loaded with the same scikit-learn version. If you ever load it in a different version, re-run this notebook to retrain.

In [15]:
def compute_baselines(Xtr):
    b = {c: float(Xtr[c].median()) for c in ("Distance_km", "Temperature_C", "Wind_Speed_kmh", "Visibility_km", "Previous_Delay_Min")}
    for c in ("Weather_Condition", "Traffic_Level", "Airline"):
        b[c] = str(Xtr[c].mode().iloc[0])
    m = float(time_to_minutes(Xtr["Scheduled_Departure"]).median())
    b["Scheduled_Departure"] = f"{int(m // 60):02d}:{int(m % 60):02d}"
    return b

cats = {"airlines": sorted(X_train["Airline"].unique()), "weather": WEATHER_CONDITIONS, "traffic": TRAFFIC_LEVELS,
        "airports": sorted(set(X_train["Origin"]) | set(X_train["Destination"]))}
bundle = {"classifier": best_pipeline, "regressor": regressor, "model_name": best_name,
          "decision_threshold": DECISION_THRESHOLD, "baselines": compute_baselines(X_train),
          "categories": cats, "sklearn_version": sklearn.__version__}
joblib.dump(bundle, MODEL_PATH)
bundle = load_model_bundle(MODEL_PATH)   # also checks the scikit-learn version
print("Saved and reloaded:", bundle["model_name"])

Saved and reloaded: Logistic Regression


## 13. Predict one flight
Edit the dictionary and re-run. Leave `Previous_Delay_Min` as `None` if the previous leg's delay isn't known before departure.

In [16]:
flight = dict(Airline="SkyJet Airways", Origin="JFK", Destination="ORD", Departure_Date="2026-12-05",
              Scheduled_Departure="18:30", Scheduled_Arrival="20:10", Distance_km=1190,
              Weather_Condition="Thunderstorm", Temperature_C=18.0, Wind_Speed_kmh=38.0,
              Visibility_km=3.5, Traffic_Level="High", Previous_Delay_Min=30)
try:
    result = predict_single(bundle, flight)
except InputValidationError as err:
    print("Fix these inputs:", *err.messages, sep="\n - ")
else:
    print(f"Predicted: {result['Predicted_Status']}  |  P(delay) = {result['Delay_Probability']:.1%}  |  "
          f"Risk: {result['Risk_Category']}  |  Estimated delay: {result['Estimated_Delay_Min']:.0f} min")
    print("(Probability is a model estimate, not a guarantee.)\n")
    for f in result["Explanation"]:
        print(f"- {f['factor']} ({f['value']} vs typical {f['typical_value']}) {f['direction']} odds of delay x{f['odds_multiplier']}")
    risk_gauge(result["Delay_Probability"]).show()
    factor_impact_chart(result["Explanation"]).show()

Predicted: Delayed  |  P(delay) = 100.0%  |  Risk: High  |  Estimated delay: 45 min
(Probability is a model estimate, not a guarantee.)

- Previous delay (30 min vs typical 3.5 min) raises odds of delay x18.76
- Weather (Thunderstorm vs typical Clear) raises odds of delay x8.11
- Wind speed (38 km/h vs typical 16 km/h) raises odds of delay x7.87
- Visibility (3.5 km vs typical 8.6 km) raises odds of delay x5.91
- Traffic level (High vs typical Medium) raises odds of delay x3.94


## 14. Batch prediction
Use any DataFrame/CSV with the required columns. Invalid rows are skipped and reported; actual-delay columns are ignored.

In [17]:
batch = pd.DataFrame([
    flight,
    dict(flight, Weather_Condition="Clear", Traffic_Level="Low", Wind_Speed_kmh=8, Visibility_km=10,
         Previous_Delay_Min=None, Scheduled_Departure="07:00", Scheduled_Arrival="08:40"),
    dict(flight, Traffic_Level="Gridlock"),      # invalid on purpose
])
batch["Flight_ID"] = ["T1", "T2", "T3"]
# batch = pd.read_csv("your_flights.csv")        # <- use your own file instead
predictions, errors = predict_dataframe(bundle, batch)
predictions.to_csv("flight_delay_predictions.csv", index=False)
print("Saved flight_delay_predictions.csv  | rejected rows:", len(errors))
display(errors)
predictions[["Flight_ID", "Origin", "Destination", "Predicted_Status", "Delay_Probability", "Risk_Category", "Estimated_Delay_Min"]]

Saved flight_delay_predictions.csv  | rejected rows: 1


,Row,Flight_ID,Issues
0,3,T3,"Traffic_Level must be one of ['Low', 'Medium',..."


,Flight_ID,Origin,Destination,Predicted_Status,Delay_Probability,Risk_Category,Estimated_Delay_Min
0,T1,JFK,ORD,Delayed,1.0000,High,44.9
1,T2,JFK,ORD,On-Time,0.0118,Low,3.3


## 15. Limitations
* Only 500 records (~100 test flights): metrics are noisy and change with the random split; CV and test rankings can disagree.
* The data is **synthetic** - relationships were written into the generator, so scores show the model recovers simulated rules, not real airline behaviour.
* Crew, maintenance, ATC, slots and cascading network effects are not modelled; probabilities are uncalibrated estimates.
* This is not live flight tracking.

**Future work:** real historical data with time-based validation, live weather/schedule APIs, probability calibration, SHAP explanations, drift monitoring.